## Selectors

In this notebook, we look at Kubernetes selectors.

- Make sure you have a Minikube Kubernetes cluster running, e.g. `minukube start`.
- Also make sure you have installed the `kubectl` tool on your computer.

## Deploy a Pod

- The Pod is defined in the YAML file `manifests/myapp.yaml`

In [1]:
!kubectl apply -f manifests/myapp.yaml

pod/myapp-pod created


## Let's look at the Pod's YAML

**Note:**

- The Pod's container is listening on `containerPort` 80.
- The Pod's `labels` are `app: myapp` and `type: front-end`.

```bash
apiVersion: v1
kind: Pod            # we are defining a Pod
metadata:
  name: myapp-pod    # the Pod's name is myapp-pod
  labels:
    app: myapp       # the Pod has two labels defined:
    type: front-end  # app = myapp and type = front-end
spec:
  containers:
  - name: nginx-container  # the name of the container (running in the Pod)
    image: nginx           # the image the container is based on
    resources:
      requests:            # minimum requested CPU/RAM for the container
        cpu: 100m
        memory: 128Mi
      limits:              # maximum allowable CPU/RAM for the container
        cpu: 250m
        memory: 256Mi    
    ports:
    - containerPort: 80    # the container is listening on port 80
```

In [2]:
#!type manifests\myapp.yaml # use this on Windows
!cat manifests/myapp.yaml # use this on Linux/Mac

apiVersion: v1
kind: Pod
metadata:
  name: myapp-pod
  labels:
    app: myapp # change this to: myapp2
    type: front-end
spec:
  containers:
  - name: nginx-container
    image: nginx
    resources:
      requests:
        cpu: 100m
        memory: 128Mi
      limits:
        cpu: 250m
        memory: 256Mi    
    ports:
    - containerPort: 80

## Deploy a Service

- The Service is defined in the YAML file `manifests/myservice.yaml`

In [3]:
!kubectl apply -f manifests/myservice.yaml

service/myservice created


## Let's look at the Pod's YAML

**Note:**

- The Service is listening on `port` 80 (on its own IP address) and redirects traffic to `targetPort` 80 (on another IP address).
- The Service's `targetPort` matches the Pod's `containerPort` above.
  - The Service will output traffic to port 80 (`targetPort`).
  - The Pod's container above is listening on port 80 (`containerPort`).
- The Service's `selector` labels match the Pod's `labels` (`app: myapp` and `type: front-end`)
  - Therefore, the Service will redirect traffic to the Pod (and its container).
  - If more Pod's had the same `labels`, the Service would load balance (using a *round robin* strategy) between all matching Pods.

```bash
apiVersion: v1
kind: Service         # we are defining a Service
metadata:
  name: myservice     # the Service's name is myservice
spec:
  ports:
  - port: 80          # the service is listening on port 80
    targetPort: 80    # the service is redirecting traffic to port 80 (on any Pod with matching labels below)
  selector:
    app: myapp        # the Service is selecting Pods that have two labels defined:
    type: front-end   # app = myapp and type = front-end
```

In [4]:
#!type manifests\myservice.yaml # use this on Windows
!cat manifests/myservice.yaml # use this on Linux/Mac

apiVersion: v1
kind: Service
metadata:
  name: myservice
spec:
  ports:
  - port: 80
    targetPort: 80
  selector:
    app: myapp
    type: front-end

## List Pods

- Notice the `IP` address of the pod.
  - Note that `pods` can be abbreviated to `po`.

In [5]:
!kubectl get po -o wide
#!kubectl get pods -o wide

NAME        READY   STATUS    RESTARTS   AGE   IP           NODE       NOMINATED NODE   READINESS GATES
myapp-pod   1/1     Running   0          24s   10.244.0.6   minikube   <none>           <none>


## List Pod's container name and containerPort

- Notice the `containerPort` the container is listening on.
  - We are using `jsonpath` to extract specific information (`name` and `containerPort`) from the JSON response.
  - See the [Jsonpath Documentation](https://kubernetes.io/docs/reference/kubectl/jsonpath) for more details.

In [6]:
!kubectl get pod myapp-pod -o jsonpath="{range .spec.containers[*]}name: {.name}, containerPort: {.ports[*].containerPort}{\"\n\"}{end}"

name: nginx-container, containerPort: 80


## List Service Endpoints (ep)

- If the Service is connected to the Pod.
  - The *Service Endpoint* will point to the Pod's IP address and Port.
- Notice the Service Endpoint (`ENDPOINTS`)
  - Matches the Pod IP address (`IP`) above.
  - Matches the container's port (`containerPort`) above (which is Port `80`).
  - If the Service was load balancing between multiple Pods, there would be more IP addresses listed under `ENDPOINTS`.
- Note that:
  - `endpoints` can be abbreviated to `ep`
  - `endpoints` has been depricated in favor of `endpointslices`

In [7]:
#!kubectl get ep myservice
!kubectl get endpoints myservice

NAME        ENDPOINTS       AGE
myservice   10.244.0.6:80   28s


- We can also use `endpointslices` to list the endpoints of a service (this replaces the depricated subcommand `endpoints`).
- The mandatory `-l` option is used to specify the search criteria.
  - `kubernetes.io/service-name` specifies we are searching for a service name.
  - `myservice` is the name of the service we are searching for.
- The output gives us:
  - The `NAME` of the service instance (i.e. the name of the service with the instance number as a suffix).
  - The `ADDRESSTYPE` of the endpoints the service is forwarding traffic to (in this case to one `IPv4` address, as opposed to a `IPv6` address).
  - The `PORTS` of the endpoints the service is forwarding traffic to (in this case to one Pod's container port `80`).
  - The `ENDPOINTS` the service is forwarding traffic to (in this one endpoint which is the IP address of one Pod).
  - The `AGE` of the service endpoints (in this case there is only one endpoint, but would be more if the service was load-balancing between multiple endpoints).

In [8]:
!kubectl get endpointslices -l kubernetes.io/service-name=myservice

NAME              ADDRESSTYPE   PORTS   ENDPOINTS    AGE
myservice-74gbt   IPv4          80      10.244.0.6   36s


## List Services

- Our Service is called `myservice`.
- Under `PORT(S)` we see it is listening on port 80.
- Under `SELECTOR` we see the label(s) it uses to select Pods to forward traffic to, i.e. Pods that have the same label(s) defined.

In [9]:
#!kubectl get svc -o wide
!kubectl get services -o wide

NAME         TYPE        CLUSTER-IP      EXTERNAL-IP   PORT(S)   AGE   SELECTOR
kubernetes   ClusterIP   10.96.0.1       <none>        443/TCP   53m   <none>
myservice    ClusterIP   10.106.144.39   <none>        80/TCP    43s   app=myapp,type=front-end


## Port forward your host machine's port 8080 to the service's port 80

- The kubernetes resources above (Pods and Services) are assigned IP addresses on Kubenetes' internal/private network.
- To access the resources from our host machine (local computer), we can define a host machine port that connects to a resource port inside the Kubernetes cluster.
  - This is called a *port-forward* and is accomplished using the command `kubectl port-forward [Resource] [HostPort]:[ResourcePort]`
    - `[Resource]` is the Kubernetes resource we want to connect to.
    - `[HostPort]` is the Port we want to use on our host machine.
    - `[ResourcePort]` is the Port we want to use on our Kubernetes resource.
  - In this case we are connecting (*port-forwarding*) port `8080` on our host machine to our Service's port `80`.
  - In turn, our Service will forward traffic to the Pod's container Port `80`.
  - The traffic flow is as follows: `Hostmachine (8080) <--> Service (80) <--> Pod's Container (80)`.
- The container is running an nginx web server on port `80` (which serves a default web page).

### Run these commands in a terminal (won't work from a notebook cell)
- If Port `8080` is already in use on your host machine, replace Port `8080` with another Port Number below.

```bash
# Port forward to the service
kubectl port-forward service/myservice 8080:80

# --- Output ---
# Forwarding from 127.0.0.1:8080 -> 80
# Forwarding from [::1]:8080 -> 80
# --------------

# Open a browser and enter the URL http://localhost:8080
# or open a new terminal and run the command below (assumes the 'curl' command is installed on your host machine):
curl http://localhost:8080

# --- Output ---
# <!DOCTYPE html>
# <html>
# <head>
# <title>Welcome to nginx!</title>
# <style>
# html { color-scheme: light dark; }
# body { width: 35em; margin: 0 auto;
# font-family: Tahoma, Verdana, Arial, sans-serif; }
# </style>
# </head>
# <body>
# <h1>Welcome to nginx!</h1>
# <p>If you see this page, the nginx web server is successfully installed and
# working. Further configuration is required.</p>
# 
# <p>For online documentation and support please refer to
# <a href="http://nginx.org/">nginx.org</a>.<br/>
# Commercial support is available at
# <a href="http://nginx.com/">nginx.com</a>.</p>
# 
# <p><em>Thank you for using nginx.</em></p>
# </body>
# </html>
# --------------

# Stop the port forward by typing Ctrl + C in the terminal

# --- Output ---
# ^C
# --------------
```

## Edit the Pod's YAML file

- Open the Pod's YAML file `manifests/myapp.yaml`.
- Change the `app` label to `myapp2`.
- Save the file.

```bash
apiVersion: v1
kind: Pod
metadata:
  name: myapp-pod
  labels:
    app: myapp2 # change this to: myapp2
    type: front-end
spec:
  containers:
  - name: nginx-container
    image: nginx
    resources:
      requests:
        cpu: 100m
        memory: 128Mi
      limits:
        cpu: 250m
        memory: 256Mi    
    ports:
    - containerPort: 80
```

## Deploy (apply) the Pod's modified YAML file

In [10]:
!kubectl apply -f manifests/myapp.yaml

pod/myapp-pod configured


## Check the Service Endpoint again

Using `endpoints`:
- Notice the Service's `ENDPOINTS` is empty (`<none>`).
  - The service's `selector` can't find any Pods with the labels `app: myapp` and `type: front-end`.
  - This is because the Pod's labels are now  `app: myapp2` and `type: front-end`.

In [11]:
#!kubectl get ep myservice
!kubectl get endpoints myservice

NAME        ENDPOINTS   AGE
myservice   <none>      2m39s


Using `endpointslices`:
- Notice the Service's `ENDPOINTS` is empty (`<unset>`).
  - The service's `selector` can't find any Pods with the labels `app: myapp` and `type: front-end`.
  - This is because the Pod's labels are now  `app: myapp2` and `type: front-end`.

In [12]:
!kubectl get endpointslices -l kubernetes.io/service-name=myservice

NAME              ADDRESSTYPE   PORTS     ENDPOINTS   AGE
myservice-74gbt   IPv4          <unset>   <unset>     2m48s


## Try to port forward to the Service again

### Run these commands in a terminal (won't work from a notebook cell)
- If Port `8080` is already in use on your host machine, replace Port `8080` with another Port Number below.
- Note that it will, by default, take 1 minute for the connection attempt to time out.

```bash
## Port forward port 8080 to the service (that's listening on port 80)
kubectl port-forward service/myservice 8080:80

# --- Output ---
# error: timed out waiting for the condition
# --------------

# If not stopped (timed out), stop the port forward by typing **Ctrl-C** in the terminal

# --- Output ---
# ^C
# --------------
```

## Edit the Pod's YAML file

- Open the Pod's YAML file `manifests/myapp.yaml`.
- Change the `app` label back to `myapp`.
- Save the file.

```bash
apiVersion: v1
kind: Pod
metadata:
  name: myapp-pod
  labels:
    app: myapp # change this to: myapp
    type: front-end
spec:
  containers:
  - name: nginx-container
    image: nginx
    resources:
      requests:
        cpu: 100m
        memory: 128Mi
      limits:
        cpu: 250m
        memory: 256Mi    
    ports:
    - containerPort: 80
```

## Deploy (apply) the Pod's modified YAML file

In [13]:
!kubectl apply -f manifests/myapp.yaml

pod/myapp-pod configured


## Check the Service Endpoint again

- Notice the Service's `ENDPOINTS` once again contains an IP address and Port.
  - The service's `selector` once again finds a Pod with the labels `app: myapp` and `type: front-end`.
  - This is because the Pod's labels are now  `app: myapp` and `type: front-end`.

In [14]:
#!kubectl get ep myservice
!kubectl get endpoints myservice

NAME        ENDPOINTS       AGE
myservice   10.244.0.6:80   3m31s


Using `endpointslices`
- Notice the Service's `ENDPOINTS` once again contains an IP address.
  - The service's `selector` once again finds a Pod with the labels `app: myapp` and `type: front-end`.
  - This is because the Pod's labels are now  `app: myapp` and `type: front-end`.

In [15]:
!kubectl get endpointslices -l kubernetes.io/service-name=myservice

NAME              ADDRESSTYPE   PORTS   ENDPOINTS    AGE
myservice-74gbt   IPv4          80      10.244.0.6   3m35s


## Delete the Service and Pod

In [16]:
!kubectl delete -f manifests/myservice.yaml
!kubectl delete -f manifests/myapp.yaml

service "myservice" deleted from default namespace
pod "myapp-pod" deleted from default namespace


## List Services and Pods

- We see that the Service and Pods have been deleted.
- Note that the service with **NAME** `kubernetes` of **TYPE** `ClusterIP` should never be deleted.
  - It's the default Kubernetes API service that exists in every cluster, and provides internal access to the Kubernetes API server.
  - Inside every Pod, Kubernetes injects environment variables and DNS entries that point to this service, that allows:
    - Pods to talk to the Kubernetes API.
    - Controllers and operators to interact with the API.
    - In-cluster clients to authenticate and manage resources.
  - For example, inside a pod, this resolves to the DNS entry https://kubernetes.default.svc which maps to the IP address and Port you see below https://IP:Port
  - This service points to the Kubernetes API server process running on your control plane node(s).

In [17]:
#!kubectl get svc
#!kubectl get po
!kubectl get services
!kubectl get pods

NAME         TYPE        CLUSTER-IP   EXTERNAL-IP   PORT(S)   AGE
kubernetes   ClusterIP   10.96.0.1    <none>        443/TCP   56m
No resources found in default namespace.
